In [1]:
import pandas as pd
import numpy as np

# Load your local CSV file
df_titanic = pd.read_csv('titanic.csv')

# Verify that the file loaded correctly
print("Dataset successfully loaded!")
print("First 5 rows:")
print(df_titanic.head())

Dataset successfully loaded!
First 5 rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0    

In [2]:
# 1. Number of records (rows) and features (columns)
print("Dataset Shape (Rows, Columns):", df_titanic.shape)

# 2. Feature names and Data Types
print("\n--- Data Types & Info ---")
print(df_titanic.info())

# 3. Target Variable Distribution
print("\n--- Target Variable (Survived) Count ---")
print(df_titanic['Survived'].value_counts())

# 4. Count of Missing Values per feature
print("\n--- Missing Values Count ---")
print(df_titanic.isnull().sum())

# 5. Duplicate Records Count
print("\nDuplicate Records:", df_titanic.duplicated().sum())

# 6. Basic Statistical Summary
print("\n--- Summary Statistics ---")
print(df_titanic.describe(include='all'))

Dataset Shape (Rows, Columns): (891, 12)

--- Data Types & Info ---
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None

--- Target Variable (Survived) Count ---
Survived
0    549
1    342
Name: count, dtype: int64

--- Missing Values Count ---
PassengerId      0
Survived         0
Pclass           0
Name

In [3]:
# Remove duplicate records if any exist
df_titanic = df_titanic.drop_duplicates()

# Verify dataset shape after cleaning
print("Shape after removing duplicates:", df_titanic.shape)

Shape after removing duplicates: (891, 12)


In [4]:
# Input features (X) and Target variable (y)
X = df_titanic.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = df_titanic['Survived']

print("Input Features (X):", list(X.columns))
print("Target Variable (y):", y.name)

Input Features (X): ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Target Variable (y): Survived


In [5]:
# Group numerical features
num_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

# Group categorical features
cat_features = ['Sex', 'Embarked']

In [8]:
from sklearn.model_selection import train_test_split

# Split into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (712, 7), X_test shape: (179, 7)


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Pipeline for Numerical Features (Impute missing with median + Standard Scaling)
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline for Categorical Features (Impute missing with mode + One-Hot Encoding)
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine both into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# 4. Fit ONLY on training data, then transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [10]:
print("Final Processed Training Set Shape (X_train_processed):", X_train_processed.shape)
print("Final Processed Testing Set Shape (X_test_processed):", X_test_processed.shape)

Final Processed Training Set Shape (X_train_processed): (712, 10)
Final Processed Testing Set Shape (X_test_processed): (179, 10)
